# Chapter 16: Multi-Agent Systems Architecture
## Notebook 01 — Agents & Messaging

We start from the atom of the system: a single **agent** with a role, tools, and memory, running a **perceive-decide-act** loop. Then we let two agents talk through typed **messages** and a shared **blackboard**.

### What you'll learn

| Topic | Section |
|-------|--------|
| What an agent is (role, tools, memory, loop) | §1 |
| The perceive-decide-act cycle | §2 |
| Typed message performatives | §3 |
| Shared state with a blackboard | §4 |

**Time estimate:** 3 hours

### Key concepts

- **Agent** — a policy plus tools and memory that acts toward a goal.
- **Performative** — the *intent* of a message (inform, request, propose…), not just its text.
- **Blackboard** — a shared workspace; powerful but a contention hazard.
- **Autonomy vs control** — more agent freedom means more capability and more failure modes.

---
*Generated by Berta AI*

## 1. The agent abstraction

An agent bundles a **role**, a set of **tools**, some **memory**, and a **policy** that maps what it sees to what it does. The policy can be a rule, a learned model, or an LLM call.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from agents import Agent, Message, Blackboard, contract_net

researcher = Agent("researcher", skill=0.9, cost=1.0)
writer = Agent("writer", skill=0.6, cost=0.4)
print(researcher, writer)

## 2. Perceive-decide-act

Every agent runs the same loop: read inputs, decide, act, update memory. Keeping the loop explicit makes the system debuggable.

In [ ]:
def step(agent, board):
    # perceive
    todo = board.read("todo", [])
    # decide (toy policy: take the first task you are skilled enough for)
    task = next((t for t in todo if agent.skill >= t["min_skill"]), None)
    # act
    if task:
        board.write(f"done:{task['id']}", agent.name)
    return task

board = Blackboard()
board.write("todo", [{"id": "t1", "min_skill": 0.5}])
print("researcher took:", step(researcher, board))
print("board:", board.facts)

## 3. Messages carry intent

A `Message` records *who* says *what kind of thing* to *whom*. Routing on the performative (not the raw text) keeps coordination logic clean.

In [ ]:
msgs = [
    Message("researcher", "writer", "inform", {"finding": "42"}),
    Message("writer", "researcher", "request", {"need": "sources"}),
]
for m in msgs:
    print(f"{m.sender} --{m.performative}--> {m.recipient}: {m.content}")

---

## Summary

An agent is a policy + tools + memory running a perceive-decide-act loop. Messages carry **intent** via performatives, and a blackboard shares state at the price of contention. These primitives compose into everything that follows.

---
*Generated by Berta AI | Berta Chapters*